In [ ]:
!pip install wikipedia
!pip install sentence_transformers

In [2]:
import re
import pandas as pd
#from datasets import load_dataset
import time
import pickle
from tqdm import tqdm

In [3]:
df=pd.read_pickle("/content/data_3.pkl")

In [4]:
df.head()

,question,answer,C
0,Harper decides to go to a nearby animal farm t...,7,First search [number of legs a horse has -Wiki...
1,"In a parking lot, There are 10 cars and 2 bike...",44,First search [number of wheels a car has -Wiki...
2,If fencing was done around a park in the shape...,43,First search [number of sides in a square -Wik...
3,"Alexander goes to school at 12 pm. On Tuesday,...",4,First search [number of hours in a school day ...
4,"At the end of the month, Hudson was checking h...",15,First search [number of days in December -Wiki...


In [8]:
df.loc[1]['answer']

44

In [ ]:
df.shape

(241, 3)

In [ ]:
import requests
import json
from bs4 import BeautifulSoup
from pprint import pprint
import wikipedia

from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity
sentence_model = SentenceTransformer("all-MiniLM-L6-v2")

from transformers import pipeline
question_answerer = pipeline("question-answering", model='distilbert-base-cased-distilled-squad')


def wikipedia_seach(query):
  language_code = 'en'

  number_of_results = 3
  headers = {
    # 'Authorization': 'Bearer YOUR_ACCESS_TOKEN',
    'User-Agent': 'YOUR_APP_NAME (YOUR_EMAIL_OR_CONTACT_PAGE)'
  }

  base_url = 'https://api.wikimedia.org/core/v1/wikipedia/'
  endpoint = '/search/page'
  url = base_url + language_code + endpoint

  search_query = query
  parameters = {'q': search_query, 'limit': number_of_results}
  response = requests.get(url, headers=headers, params=parameters)
  response = json.loads(response.text)

  searched_titles=[]
  for page in range(len(response['pages'])):
    display_title = response['pages'][page]['title']
    searched_titles.append(display_title)
    response['pages'][page]["article_url"] = 'http://en.wikipedia.org/?curid=' + str(response['pages'][page]['id'])
    response['pages'][page]['excerpt'] = BeautifulSoup(response['pages'][page]['excerpt']).get_text() #cleaning excerpts and saving

  excerpt_text=search_query+": "+"\n".join([page['excerpt'] for page in response['pages']]) #excerpt text

  if(len(searched_titles)!=0):
    #go to page whose title+ excerpt is most similar
    # Encode search query
    query_embedding = sentence_model.encode(search_query)
    page_texts = [page['title'] + ": " + page['excerpt'] for page in response['pages']] #page title + excerpt
    page_embeddings = sentence_model.encode(page_texts)
    # Calculate similarity scores for each page
    similarities = cosine_similarity([query_embedding], page_embeddings)[0]

    title_score_dict = dict(zip(searched_titles, similarities))

    # Get the index of the page with the highest similarity
    max_similarity_index = similarities.argmax()

    # Output the page id with the highest similarity
    max_similarity_page_id = response['pages'][max_similarity_index]['id']

    try:
      a=wikipedia.page(pageid=max_similarity_page_id)
      full_content=a.content
      return title_score_dict,excerpt_text,full_content
    except:
      return {},"",""
  else:
    return {},"",""

def QA(ques, context):
  result = question_answerer(question=ques,context=context)
  return result['answer'], result['score']

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

In [ ]:
#google search

import requests
import json

url = "https://google.serper.dev/search"

payload = json.dumps({
  "q": "number of cards in a deck"
})
headers = {
  'X-API-KEY': '73bcb6abc9a600474178aa08cee6a7e32e76de6f',
  'Content-Type': 'application/json'
}

response = requests.request("POST", url, headers=headers, data=payload)

print(response.text)

{"searchParameters":{"q":"number of cards in a deck","type":"search","engine":"google"},"answerBox":{"snippet":"A \"standard\" deck of playing cards consists of 52 Cards in each of the 4 suits of Spades, Hearts, Diamonds, and Clubs. Each suit contains 13 cards: Ace, 2, 3, 4, 5, 6, 7, 8, 9, 10, Jack, Queen, King. Modern decks also usually include two Jokers.","snippetHighlighted":["52 Cards"],"title":"standard deck playing card games | Wiki - BoardGameGeek","link":"https://boardgamegeek.com/wiki/page/standard_deck_playing_card_games"},"organic":[{"title":"[PDF] Standard Deck of Cards","link":"https://intranet.missouriwestern.edu/cas/wp-content/uploads/sites/17/2020/05/Standard-Deck-of-Cards.pdf","snippet":"There are 52 cards in a standard deck. There are 4 suits; Diamonds, Clubs, Hearts, Spades. “Face Cards” include Jacks, Queens, and Kings.","position":1},{"title":"Standard 52-card deck - Wikipedia","link":"https://en.wikipedia.org/wiki/Standard_52-card_deck","snippet":"In other region

In [ ]:
response = json.loads(response.text)

In [ ]:
response.keys()

dict_keys(['searchParameters', 'answerBox', 'organic', 'peopleAlsoAsk', 'relatedSearches'])

In [ ]:
response['answerBox']['snippet']

'A "standard" deck of playing cards consists of 52 Cards in each of the 4 suits of Spades, Hearts, Diamonds, and Clubs. Each suit contains 13 cards: Ace, 2, 3, 4, 5, 6, 7, 8, 9, 10, Jack, Queen, King. Modern decks also usually include two Jokers.'

In [ ]:
for i in response['organic']:
  print(i['snippet'])

There are 52 cards in a standard deck. There are 4 suits; Diamonds, Clubs, Hearts, Spades. “Face Cards” include Jacks, Queens, and Kings.
In other regions, such as Spain and Switzerland, the traditional standard pack comprises 36, 40 or 48 cards. Cards from a standard, English or Anglo-American ...
How many cards are in a deck of cards without Jokers? https://ulearnmagic.com/how-many ...
There are 52 cards in a full pack of playing cards (excluding jokers) – There are 52 weeks in a year. Finally, the sum of the values of the 52 ...
A 'standard' deck of playing cards has 52, being 4 suits each of Ace, 2, 3, 4, 5, 6, 7, 8, 9, 10, Jack, Queen, King. A pack may also be ...
A standard deck of cards contains 52 number of cards of 4 different suits namely Clubs, Spades, Hearts and Diamonds. Each suit contains 13 ...
This is a basic video on a deck of playing cards, answering How many cards are in a deck ...
But exactly how many cards are in a deck? The right answer is – it depends. Most of th

In [ ]:
## CLEANING

In [ ]:
row=0
print(df.loc[row]['question'])
print(df.loc[row]['C'])
print(df.loc[row]['answer'])

Harper decides to go to a nearby animal farm that has a total of 11 animals. He is standing in a queue near the entry gate, from the gate he could only see legs of the animals in the farm and he counted the total number of legs to be 30. He already knows that there are only ducks and horses in this farm. From this information find the number of ducks
First search [number of legs on a duck -Wiki-> y1]. Ducks have [y1 -QA(How many legs on a duck?)-> y2] legs each.Then search [number of legs on a horse -Wiki-> y3]. Horses have [y3 -QA(How many legs on a horse?)-> y4] legs each.There are [11 - x = y5] horses in the farm. Each duck has y2 legs, and each horse has y4 legs. So, the total number of legs from ducks is [y2*x = y6] and from horses is [y4*y5 = y7].The total number of animal legs in the farm is [y6 + y7 = y8]. The given number of legs is 30. Hence, y8 = 30.The answer is y5.
7


In [ ]:
pattern_rows = df[df['C'].str.contains(r'is \[y3\.\s*=\s*y4\] The answer is y4', regex=True)]
pattern_rows

,Q,A,W,C


In [ ]:
df['C'] = df['C'].str.replace(r'is \[y3\.\s*=\s*y4\] The answer is y4', 'is y3', regex=True)

### EXPERIMENT

In [ ]:
row=1
print(df.loc[row]['question'])
print(df.loc[row]['C'])
print(df.loc[row]['answer'])

In a parking lot, There are 10 cars and 2 bikes. Find out the number of wheels in that parking lot.
First search [number of wheels in a car -Wiki-> y1]. A car has [y1 -QA(How many wheels in a car?)-> y2] wheels each.Then search [number of wheels on a bike -Wiki-> y3]. A bike has [y3 -QA(How many wheels on a bike?)-> y4] wheels each.There are [10 = x] cars and [2 = y] bikes in the parking lot. Each car has y2 wheels, and each bike has y4 wheels. So, the total number of wheels from cars is [y4*x = y6] and from bikes is [y4*y = y7].The total number of wheels in the parking lot is [y6 + y7 = y8]. The answer is y8.
44


In [ ]:
text=df.iloc[row]['C']
matches = re.findall(r'\[(.*?)\]', text)
matches

['number of wheels in a car -Wiki-> y1',
 'y1 -QA(How many wheels in a car?)-> y2',
 'number of wheels on a bike -Wiki-> y3',
 'y3 -QA(How many wheels on a bike?)-> y4',
 '10 = x',
 '2 = y',
 'y4*x = y6',
 'y4*y = y7',
 'y6 + y7 = y8']

In [ ]:
def process_list_tuple(rep): ## DECISION HERE
  if(str(type(rep[0]))!="<class 'tuple'>"): #if wikipedia article comes as in query, return empty string as it is anyways invalid
    return ""
  #for QA result, best score string returned
  non_empty_rep = [item for item in rep if item]  # Filter out empty tuples
  if non_empty_rep:
    rep = max(non_empty_rep, key=lambda x: x[1])[0]  # Get the string with the highest score
  else:
    rep = ""  # If all tuples are empty, return an empty string
  return str(rep)


def get_variables(text):
  matches = re.findall(r'\[(.*?)\]', text)
  variables={}
  wiki_searches=[]
  errors=[]

  def replace_variable(match):
    #print(variables)
    rep = variables.get(match.group(), match.group())
    if isinstance(rep, list):
        rep=process_list_tuple(rep)
    return rep

  for instruction in matches:
    print('at ', instruction)
    #---WIKI TOOL---
    if(instruction.find("-Wiki")!=-1):
      query,ans=instruction.split(" -Wiki-> ") #for in query variables
      in_query_var=re.search(r'y\d+', query)
      if(in_query_var):
        #print("IN QUERY VAR",query)
        query=re.sub(r'y\d+', replace_variable, query)
        #print(query)

      title_score_dict,excerpt_text,full_content=wikipedia_seach(query)#wikipedia search

      if title_score_dict: #SAVING IN VARIABLES HERE
        #print('HERE updating varibale of wiki answer',ans.strip())
        variables[ans.strip()] = [excerpt_text, full_content]#replaces the answer variable with extracted content
        wiki_searches.append(title_score_dict)
      else:
        errors.append(f"Error: Wikipedia search unsuccessful for query: '{query}'")
        continue

    #---QA TOOL---
    elif(instruction.find("-QA")!=-1):
      q,a=instruction.split("->")
      result = re.search(r'\((.*?)\)', q)
      if result:
          question = result.group(1)
      else:
          errors.append("Error in finding question")
          continue

      pre=q.split(" -QA")[0]
      context_exc=""
      full_context=""
      if (pre.find("+")!=-1): #if multiple contexts need to be appended

        vars=pre.split("+")
        for i in vars:
          if(i.strip() in variables.keys()):
            if((str(type(variables[i.strip()][0]))!="<class 'tuple'>")):
              context_exc=variables[i.strip()][0] #if variable came from wiki search, it would be a string else it would be a tuple
              full_context=variables[i.strip()][1]
            else:
              t=process_list_tuple(variables[i.strip()])
              #print('-----t',t)
              context_exc=t #if variable came from wiki search, it would be a string else it would be a tuple
              full_context=t
          else:
            errors.append(f"Error: Context variable '{i.strip()}' not found")
            continue

      else: #single context QA
        context_var=pre.strip()
        if context_var in variables.keys():
          if((str(type(variables[context_var][0]))!="<class 'tuple'>")):
            context_exc=variables[context_var][0] #if variable came from wiki search, it would be a string else it would be a tuple
            full_context=variables[context_var][1]
          else:
            t=process_list_tuple(variables[context_var])
            context_exc=t #if variable came from wiki search, it would be a string else it would be a tuple
            full_context=t
        else:
          errors.append(f"Error: Context variable '{context_var}' not found")
          continue

      execerpt_run=QA(question, context_exc)
      if(execerpt_run[-1]<0.75):
        full_run=QA(question, full_context)
      else:
        full_run=()

      #print('Updating variable',a.strip())
      variables[a.strip()]=[execerpt_run,full_run]

    #----MATH TOOL----
    else:
      print('going to match tool')
      all_eqs=[instruction]
      equation_variables = re.findall(r'y\d+', instruction)
      for var in equation_variables:
        if(var in variables.keys()):
          var_ans=process_list_tuple(variables[var])
          numbers = re.findall(r'\d+(?:\.\d+)?', var_ans)#we try searching for numbers
          if(len(numbers)>0):
            all_eqs.append(var+" = "+str(numbers[0])) #making equation for previously retrieved numerical values ### DECISON HERE
            errors.append(f"WARNING: '{str(numbers)}'numbers in var ans")
          else:
            print('var ans is not a numeric',var_ans)
            errors.append(f"Error: '{var_ans}' not numeric")

      print(all_eqs)
      list_eq=[]
      for i in all_eqs:
        eq=sympify("Eq(" + i.replace("=", ",") + ")")
        list_eq.append(eq)

      result =solve(list_eq)
      print(result)
      for i in result.keys():
        if(str(i) not in variables.keys()):
          #print(i)
          variables[str(i)]=result[i]


  return variables,wiki_searches,errors


In [ ]:
v,w,e=get_variables(text)

at  Catching Fire Suzanne Collins -Wiki-> y1
at  y1 -QA(What was the age of Katniss Everdeen at that time ?)-> y2
at  y2 + 6 = y3
going to match tool
['y2 + 6 = y3', 'y2 = 1962']
{y2: 1962, y3: 1968}


In [ ]:
e

["WARNING: '['1962']'numbers in var ans"]

In [ ]:
v

In [ ]:
##sympy setup
!git clone https://github.com/sympy/sympy.git
import os
os.chdir('sympy/')
os.getcwdb()
!python -m pip install -e .